# 01 — Download & Prepare the Ecommerce Customer Support Dataset

**Project:** Ecommerce-LLM-Finetuning
**Goal of this notebook:**
1. Install lightweight dependencies
2. Download a free, public ecommerce/customer-support dataset from Hugging Face
3. Inspect its structure
4. Remove duplicate records
5. Split into train / validation / test
6. Save raw and split data to disk for the next notebooks

**Dataset used:** [`bitext/Bitext-customer-support-llm-chatbot-training-dataset`](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)
This is a free, open, Hugging-Face-hosted dataset of ~27k instruction/response pairs
covering customer support intents (orders, shipping, refunds, payments, accounts, etc.)
that map directly onto ecommerce support use cases. It requires no authentication token.

> If Hugging Face rate-limits or this dataset becomes unavailable, a fallback
> synthetic-but-realistic sample is generated at the bottom of this notebook so
> every downstream notebook still runs end-to-end.

**Best practices demonstrated:** reproducible seeds, explicit logging, no hard-coded
paths (uses `src/config.py`), defensive fallback if network/dataset access fails.


In [1]:
# ---------------------------------------------------------------------------
# 1. Install dependencies (Colab-safe, quiet install)
# ---------------------------------------------------------------------------
!pip install -q datasets==2.20.0 pandas==2.2.2 scikit-learn==1.5.0 huggingface_hub==0.24.0


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Program Files\Python313\Scripts\pip.exe\__main__.py", line 2, in <module>
  File "C:\Users\Asus\AppData\Roaming\uv\python\cpython-3.10-windows-x86_64-none\Lib\re.py", line 125, in <module>
    import sre_compile
  File "C:\Users\Asus\AppData\Roaming\uv\python\cpython-3.10-windows-x86_64-none\Lib\sre_compile.py", line 17, in <module>
    assert _sre.MAGIC == MAGIC, "SRE module mismatch"
AssertionError: SRE module mismatch


In [2]:
# ---------------------------------------------------------------------------
# 2. Imports & setup
# ---------------------------------------------------------------------------
import os
import sys
import json
import logging
import random

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("01_download_dataset")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------------------------------------------------------------------------
# 3. Paths — mirrors src/config.py so this notebook is portable to Colab
#    (clone the repo, or these folders are created relative to CWD)
# ---------------------------------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
DATA_SAMPLE_DIR = os.path.join(PROJECT_ROOT, "data", "sample")

for d in [DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_SAMPLE_DIR]:
    os.makedirs(d, exist_ok=True)

logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"Raw data dir: {DATA_RAW_DIR}")


c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-02 11:50:36,148 | INFO | Project root: c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning
2026-07-02 11:50:36,150 | INFO | Raw data dir: c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\raw


In [3]:
# ---------------------------------------------------------------------------
# 4. Download the dataset from Hugging Face
# ---------------------------------------------------------------------------
DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

def download_hf_dataset(name: str):
    """Attempt to download the primary dataset from the Hugging Face Hub.

    Args:
        name: HF dataset repo id.

    Returns:
        pandas.DataFrame with the raw dataset, or None if download failed.
    """
    try:
        logger.info(f"Downloading dataset: {name}")
        ds = load_dataset(name, split="train")
        df = ds.to_pandas()
        logger.info(f"Downloaded {len(df):,} rows, columns: {list(df.columns)}")
        return df
    except Exception as e:
        logger.error(f"Failed to download {name}: {e}")
        return None

raw_df = download_hf_dataset(DATASET_NAME)


2026-07-02 11:50:36,186 | INFO | Downloading dataset: bitext/Bitext-customer-support-llm-chatbot-training-dataset
2026-07-02 11:50:43,622 | INFO | Downloaded 26,872 rows, columns: ['flags', 'instruction', 'category', 'intent', 'response']


In [4]:
# ---------------------------------------------------------------------------
# 5. Fallback: synthetic-but-realistic ecommerce support sample
#    (only triggers if the HF download above failed, e.g. no internet in
#    this environment). Guarantees every notebook downstream still runs.
# ---------------------------------------------------------------------------
def build_fallback_dataset(n: int = 500) -> pd.DataFrame:
    """Builds a small synthetic ecommerce support dataset as a safety net.

    Args:
        n: number of synthetic rows to generate.

    Returns:
        pandas.DataFrame with columns ['instruction', 'category', 'intent', 'response'].
    """
    logger.warning("Using fallback synthetic dataset generator.")
    templates = [
        ("Where is my order #{oid}?", "ORDER", "track_order",
         "Your order #{oid} has shipped and is expected to arrive within 3-5 business days. "
         "You can track it in real time using the link in your confirmation email."),
        ("I want to return the item from order #{oid}.", "RETURN", "return_item",
         "You can start a return for order #{oid} from the 'My Orders' page within 30 days "
         "of delivery. Once we receive the item, your refund will be processed within 5-7 days."),
        ("My payment for order #{oid} failed, what do I do?", "PAYMENT", "payment_issue",
         "We're sorry about that. Please check that your card details are correct and try again, "
         "or use an alternate payment method. No charge was made for the failed attempt."),
        ("Do you have any active coupon codes?", "PROMOTION", "coupon_inquiry",
         "You can find current promotions on our Deals page. Sign up for our newsletter to get "
         "exclusive coupon codes sent directly to your inbox."),
        ("How do I update my shipping address for order #{oid}?", "ACCOUNT", "update_address",
         "As long as order #{oid} hasn't shipped yet, you can update the address from your "
         "order details page. Once shipped, please contact support for rerouting options."),
        ("I never received a refund for order #{oid}.", "REFUND", "missing_refund",
         "Refunds for order #{oid} typically take 5-7 business days to appear. If it has been "
         "longer, please share your order ID so we can escalate this for you."),
    ]
    rows = []
    for i in range(n):
        instr, cat, intent, resp = templates[i % len(templates)]
        oid = 100000 + i
        rows.append({
            "instruction": instr.format(oid=oid),
            "category": cat,
            "intent": intent,
            "response": resp.format(oid=oid),
        })
    return pd.DataFrame(rows)

if raw_df is None:
    raw_df = build_fallback_dataset(n=500)

print(raw_df.shape)
raw_df.head()


(26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [5]:
# ---------------------------------------------------------------------------
# 6. Inspect dataset structure
# ---------------------------------------------------------------------------
logger.info("Dataset info:")
raw_df.info()

print("\nColumn names:", list(raw_df.columns))
print("\nMissing values per column:")
print(raw_df.isnull().sum())

print("\nSample rows:")
raw_df.sample(min(5, len(raw_df)), random_state=SEED)


2026-07-02 11:50:43,773 | INFO | Dataset info:


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB

Column names: ['flags', 'instruction', 'category', 'intent', 'response']

Missing values per column:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Sample rows:


,flags,instruction,category,intent,response
9329,BLZ,I can't talk with a human agent,CONTACT,contact_human_agent,Thank you for reaching out! I'm sorry to hear ...
4160,BLMZ,I have got to locate hte bills from {{Person N...,INVOICE,check_invoice,I'll get right on it! I understand your need t...
18500,BCELM,"I cannot pay, help me to inform of a problem w...",PAYMENT,payment_issue,I apologize for the inconvenience caused by th...
8840,BL,I want help speaking to customer service,CONTACT,contact_customer_service,Grateful for your contact! I get the sense tha...
5098,BLZ,I try to see th accepted payment options,PAYMENT,check_payment_methods,Thank you for reaching out! I completely under...


In [6]:
# ---------------------------------------------------------------------------
# 7. Save the untouched raw dataset (audit trail — never overwrite this)
# ---------------------------------------------------------------------------
raw_path = os.path.join(DATA_RAW_DIR, "ecommerce_support_raw.csv")
raw_df.to_csv(raw_path, index=False)
logger.info(f"Saved raw dataset -> {raw_path}")


2026-07-02 11:50:44,281 | INFO | Saved raw dataset -> c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\raw\ecommerce_support_raw.csv


In [7]:
# ---------------------------------------------------------------------------
# 8. Remove duplicate records
#    We dedupe on the instruction/question text (and response, if present)
#    to avoid leaking near-identical examples across train/val/test splits.
# ---------------------------------------------------------------------------
text_col_candidates = [c for c in raw_df.columns if c.lower() in
                        ("instruction", "question", "text", "utterance")]
resp_col_candidates = [c for c in raw_df.columns if c.lower() in
                        ("response", "answer", "output")]

TEXT_COL = text_col_candidates[0] if text_col_candidates else raw_df.columns[0]
RESPONSE_COL = resp_col_candidates[0] if resp_col_candidates else raw_df.columns[-1]

logger.info(f"Using TEXT_COL='{TEXT_COL}', RESPONSE_COL='{RESPONSE_COL}'")

before = len(raw_df)
dedup_df = raw_df.drop_duplicates(subset=[TEXT_COL, RESPONSE_COL]).reset_index(drop=True)
dedup_df = dedup_df.dropna(subset=[TEXT_COL, RESPONSE_COL]).reset_index(drop=True)
after = len(dedup_df)

logger.info(f"Removed {before - after:,} duplicate/empty rows ({before:,} -> {after:,})")


2026-07-02 11:50:44,321 | INFO | Using TEXT_COL='instruction', RESPONSE_COL='response'
2026-07-02 11:50:44,397 | INFO | Removed 0 duplicate/empty rows (26,872 -> 26,872)


In [8]:
# ---------------------------------------------------------------------------
# 9. Train / validation / test split (80 / 10 / 10)
# ---------------------------------------------------------------------------
train_df, temp_df = train_test_split(dedup_df, test_size=0.2, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, shuffle=True)

logger.info(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

assert len(train_df) + len(val_df) + len(test_df) == len(dedup_df), "Split sizes do not add up!"


2026-07-02 11:50:44,427 | INFO | Train: 21,497 | Val: 2,687 | Test: 2,688


In [9]:
# ---------------------------------------------------------------------------
# 10. Save processed splits + a small sample for quick inspection/demos
# ---------------------------------------------------------------------------
train_path = os.path.join(DATA_PROCESSED_DIR, "train.csv")
val_path = os.path.join(DATA_PROCESSED_DIR, "val.csv")
test_path = os.path.join(DATA_PROCESSED_DIR, "test.csv")
sample_path = os.path.join(DATA_SAMPLE_DIR, "sample_20_rows.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)
dedup_df.sample(min(20, len(dedup_df)), random_state=SEED).to_csv(sample_path, index=False)

logger.info(f"Saved train -> {train_path}")
logger.info(f"Saved val   -> {val_path}")
logger.info(f"Saved test  -> {test_path}")
logger.info(f"Saved sample-> {sample_path}")

# Persist which columns downstream notebooks should use
with open(os.path.join(DATA_PROCESSED_DIR, "column_map.json"), "w") as f:
    json.dump({"text_col": TEXT_COL, "response_col": RESPONSE_COL}, f, indent=2)

print("Notebook 01 complete. Proceed to 02_data_exploration.ipynb")


2026-07-02 11:50:44,862 | INFO | Saved train -> c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\processed\train.csv
2026-07-02 11:50:44,863 | INFO | Saved val   -> c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\processed\val.csv
2026-07-02 11:50:44,864 | INFO | Saved test  -> c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\processed\test.csv
2026-07-02 11:50:44,865 | INFO | Saved sample-> c:\Users\Asus\Downloads\Ecommerce-LLM-Finetuning\Ecommerce-LLM-Finetuning\data\sample\sample_20_rows.csv


Notebook 01 complete. Proceed to 02_data_exploration.ipynb
